# Section 0: Repository Layout

- There are 4 **shell scripts**:
    - Configure physics.
    - Run physics executable.
    - Handle file i/o.
- There is one **python notebook**, this is where you will work on the hdf5 analysis.
- There are 5 **directories**:
    - `documentation`, home of useful READMEs. 
    - `includes`, local copies of necessary software.
    - `inputs`, local copies of files providing information about beam and neutrino physics relevent for DUNE.
    - `outputs`, where the files that you make go.
    - `super_secret_solutions`, where the answers live. Please don't cheat, that would be no fun :)

You can use the `/data` directory as you did before!

# Section 1: Neutrino Event Generation

**This was the `run-genie` box in the ND sim / reco diagram.**

**Can be tricky to configure, requires knowledge of:**
- Beam flux.
- Beam position.
- Detector geometry (positions / materials).
- Neutrino interaction models.
- Neutrino cross-sections.

**Can be slow to run!**
- Depends on size / composition of geometry.
- Depends on the amount of POT being simulated.

**Outputs a list of particles produced for each neutrino interaction with quantities like:**
- Particle type (pdg).
- Position.
- Momentum.

In [ ]:
# From a terminal, when inside /home/jovyan, run:
./run_genie.sh

In [ ]:
# To check success, after seeing:


# ALL DONE!!!


# Check the outputs directory, you should see two new files.

!ls -ltr /home/jovyan/outputs

# Section 2: Particle Propagation

**This was the `run-edep-sim` box in the ND sim / reco diagram.**

**`edep-sim` is a wrapper for Geant4.**

**To configure, it requires:**
- Detector geometry.
- `gtrac` file from `run-genie` step (configured in a "macro file", `*.mac`).
- Number of neutrino interactions (extracted from the `gtrac` file apriori).

**For each of the particles produced by GENIE, outputs information including:**
- Energy deposits along the particle's path.
- Secondary interactions / secondary particles.

In [ ]:
# From a terminal, when inside /home/jovyan, run:
./run_edep_sim.sh

In [ ]:
# To check success, after seeing:


# ALL DONE!!!


# Check the outputs directory, you should see one new file.

!ls -ltr /home/jovyan/outputs

# Section 3: Spill building

**This was the `run-spill-build` box in the ND sim / reco diagram.**

**To configure, it requires:**
- Information about the beam design:
    - Intensity it can run, POT per spill.
    - Time between adjacent spills, spill period.
- `root` file from `run-edep-sim` step. 

**Outputs an updated `root` with modified timing.**

**Can also be used to overlay two samples together.**

In [ ]:
# From a terminal, when inside /home/jovyan, run:
./run_spill_build.sh

In [ ]:
# To check success, after seeing:


# ALL DONE!!!


# Check the outputs directory, you should see one new file.

!ls -ltr /home/jovyan/outputs

# Section 4: Root to HDF5 Conversion.

**This was the `run-convert2h5` box in the ND sim / reco diagram.**

**To configure, it requires only the output of `run-spill-build` step.**

Why? So we can perform analysis using what we've already learned about `numpy` and other python tools!

In [ ]:
# Open run_convert2h5.sh, and configure!

#!/usr/bin/env bash


# Specify the run number.
export ND_PRODUCTION_RUN_NUMBER=__CHANGE_ME__


# Configure the input / output file handling. 
export ND_PRODUCTION_OUT_DIR=__CHANGE_ME__
export ND_PRODUCTION_OUT_FILE=__CHANGE_ME__
export ND_PRODUCTION_IN_FILE=__CHANGE_ME__


# Necessary due to ROOT versioning :shrug:.
export CPATH=$EDEPSIM/include/EDepSim:$CPATH


# Execute the conversion script. Only requires an input file path and an output file path.
./includes/convert_edepsim_roottoh5.py __CHANGE_ME__


echo ""
echo ""
echo ""
echo "ALL DONE!!!"
echo ""
echo ""
echo ""

In [ ]:
# From a terminal, when inside /home/jovyan, run:
./run_conver2h5.sh

In [ ]:
# To check success, after seeing:


# ALL DONE!!!


# Check the outputs directory, you should see one new file.

!ls -ltr /home/jovyan/outputs

## Section 5: Analysis of NDLAr HDF5

Now we'll look at the file `hdf5` file that we made and do some physics!

Need to import a few useful packages, including some for working with hdf5 files and for plotting.

In [ ]:
import h5py
import matplotlib
import matplotlib.pyplot as plt
import math
import numpy as np

In [ ]:
f_in_path = ""
f_in = h5py.File(f_in_path, 'r')


print('\n----------------- File content -----------------------')
print('File:', f_in_path, "has", len(f_in.keys()), "datasets...")
for key in f_in.keys():
    print(f"Dataset {key:<12} contains {len(f_in[key]):>10} entries")
print('------------------------------------------------------\n')

## HDF5 Datasets / Structured Arrays:

A structured array is a NumPy array where each element has multiple named fields — like a table/spreadsheet row, where each row has named columns of potentially different types.

- `mc_hdr`, one entry per simulated neutrino, information from GENIE.
- `vertices`, one entry per simulated neutrino.
- `mc_stack`, one entry per particle produced by GENIE and propagated by edep-sim.
- `segments`, one entry per energy deposit.
- `trajectories`, one entry per true "particle trajectory".

Can look at the variables available in each dataset:

In [ ]:
# Look at the structure of one of the datasets, picking out the names of the columns.
f_in["mc_hdr"].dtype.names

In [ ]:
print(f_in["mc_hdr"]["event_id"])
print("\n\nUnique values:")
print(np.unique(f_in["mc_hdr"]["event_id"]))

# event_id indexes the spill.

## Answering a physics question like an experimental particle physicist:

**Can I make a plot to show this? A PhD in experimental particle physics is a PhD in plot making.**

**Tips:**
- Does the variable exist (check for documentation `/home/jovyan/documentation`).
- Do I understand the variable? Units, range + region of interest (axis limits), level of statistics (number of bins).
- How can I best present the information? 2D histogram, scatter, line graph?

## Physics Question 1:

### What does the energy spectrum of the neutrino beam look like?

**Hints:**
- Produce a 1D histogram of the true neutrino energy.
- Use the `mc_hdr` dataset.

In [ ]:
mc_hdr = f_in["mc_hdr"]

def mev_to_gev(e_in_mev):
    return e_in_mev/1000.

plt.hist(mev_to_gev(mc_hdr["Enu"]), bins=10)
plt.xlabel("True Neutrino Energy (GeV)")
plt.ylabel("Interactions")

In [ ]:
plt.hist(mev_to_gev(mc_hdr["Enu"]), range=[0,8], bins=10)
plt.xlabel("True Neutrino Energy (GeV)")
plt.ylabel("Interactions")

## Physics Question 2:

### a) What is the macro-timing structure of the beam?
### b) What is the micro-timing structure of the beam?

**Hints:**
- Can use the neutrino interaction times, showing the difference in time between spills for macro-timinig.
- For micro-timing, we want to know the duration of each spill. You can use the time between spills to help.
- Use the `mc_hdr` dataset.

In [ ]:
# Part a)

#def some_function(some_parameter):
#    return some_value

#plt.hist()

# Part b)

#def some_function(some_parameter):
#    return some_value

#plt.hist()

## Physics Question 3:

### How does the type of neutrino interaction relate to the interaction inelasticity?

**Hints:**
- Consider each of the identifying interaction types which come from GENIE.
- Utilise masking in `numpy`.
- Use a stacked histogram.

In [ ]:
#def some_function(some_parameter):
#    return some_value

#plt.hist()

## Physics Question 4:

### What is the flavour composition of the beam?

**Hints:**
- How do the fractions of muon neutrinos / electron neutrinos, right-sign / wrong-sign, compare?
- Utilise `pdg` values.

In [ ]:
#def some_function(some_parameter):
#    return some_value

#plt.hist()

## Physics Question 5:

### How many interactions are there per spill?

**Hints:**
- Use the fact that `event_id` labels the spill.

In [ ]:
#def some_function(some_parameter):
#    return some_value

#plt.hist()

## Physics Question 6:

### a) Produce an "Event Display" that can show the positions of all of the activity from a single spill in 3D.
### b) Produce an "Event Display" that can show the positions of all of the activity from a single neutrino interaction in 2, 2D projections (x vs.z and y vs. z).

**Hints:**
- Use the `segments` array.
- You can get information about the position and size of each energy deposition.
- Define a function which takes a spill ID and draws the display.
- I've given you a helper function to get the rough boundaries of NDLAr, you can use this to set the axis ranges.
    - x - horizonal, transverse to the beam direction.
    - y - vertical, transverse to the beam direction.
    - z - in the beam direction.
- Interactions are labelled by `vertex_id`. 

In [ ]:
# Helper function - get the rough edges of the active LAr region in NDLAr.
def get_rough_active_region(dimension):
    if dimension=="x": return [-360.0,360.0]
    if dimension=="y": return [-220.0, 84.0]
    if dimension=="z": return [ 410.0,920.0]
    raise "dimension must be x, y or z"

# a)
#def some_function(some_parameter):
#    return some_value

In [ ]:
# a)
# Call my event display function
# some_function(some_parameter)

In [ ]:
# b)
#def some_function(some_parameter):
#    return some_value

In [ ]:
# b)
# Call my event display function
# some_function(some_parameter)